In [1]:
%load_ext autoreload
%autoreload 2

# Identify whether a CUDA-enabled GPU is available
import torch
import pickle
import os
import sys
import pandas as pd
import numpy

# Optionally update sys.path for this notebook to import project code from parent directory.
sys.path.append("../..")

device = "cuda"

# data_home = "../../artefacts/data"
data_home = "../../../eeg_challenge/artefacts/data"

with open(os.path.join(data_home, "train_dataset_5sec.pkl"), "rb") as f:
    train_dataset = pickle.load(f)

with open(os.path.join(data_home, "valid_dataset.pkl"), "rb") as f:
    valid_dataset = pickle.load(f)

meta_information = train_dataset.get_metadata()
SFREQ = 100

In [2]:
meta_information_valid = valid_dataset.get_metadata()
default_rmse_valid = meta_information_valid['target'].std()
default_rmse_train = meta_information['target'].std()

print(f"Default std on train: {default_rmse_train:.4f}, on valid: {default_rmse_valid:.4f}")

Default std on train: 0.4088, on valid: 0.4054


In [3]:
from neurosned.foundation.parser import SFPParser
from neurosned.foundation.pseudo_devices import PseudoDevice
from neurosned.foundation.channels_splitter import DeviceSplitter


sfp_path = "GSN-HydroCel-129.sfp"
parser = SFPParser(sfp_path)
region_mapping = parser.channel_id_to_region()
print(f"Original region distribution: \n{pd.Series(region_mapping).value_counts().to_dict()}")

uniform_receipt = {"C": 8, 'F': 8, "O":8, "P": 7, "TL": 5, 'TR': 5}
splitter = DeviceSplitter(region_mapping, random_seed=42)
careful_pseudo_devices = splitter.uniform_split(uniform_receipt, n_devices=3, repeat=False)
random_pseudo_devices = splitter.random_split(n_devices=3, n_channels_per_device=41)
worst_pseudo_devices = splitter.worst_split(n_devices=3, n_channels_per_device=41)

# Test distributions of channels by region for each pseudodevice (uniform_split)
for name, devices in [
    ("uniform_split", careful_pseudo_devices),
    # ("random_split", random_pseudo_devices),
    # ("worst_split", worst_pseudo_devices)
]:
    for i, device in enumerate(devices):
        print(f"Pseudodevice ({name}) {i}:\n{device}\n")

Original region distribution: 
{'C': 24, 'F': 24, 'O': 24, 'P': 23, 'TL': 17, 'TR': 17}
Pseudodevice (uniform_split) 0:
<PseudoDevice(n_channels=41, reference_channel=118)>
  C: 8 -> 28, 31, 110, 111, 116, 117, 118, 128
  F: 8 -> 2, 7, 9, 11, 13, 15, 23, 24
  O: 8 -> 58, 67, 72, 73, 81, 87, 88, 89
  P: 7 -> 52, 57, 77, 84, 86, 91, 95
  TL: 5 -> 34, 37, 43, 44, 49
  TR: 5 -> 99, 101, 112, 115, 120

Pseudodevice (uniform_split) 1:
<PseudoDevice(n_channels=41, reference_channel=103)>
  C: 8 -> 0, 12, 26, 27, 35, 47, 103, 122
  F: 8 -> 1, 8, 10, 16, 18, 21, 123, 126
  O: 8 -> 64, 68, 70, 74, 76, 83, 90, 93
  P: 7 -> 30, 36, 53, 54, 59, 62, 78
  TL: 5 -> 32, 40, 46, 48, 55
  TR: 5 -> 96, 100, 102, 109, 121

Pseudodevice (uniform_split) 2:
<PseudoDevice(n_channels=41, reference_channel=124)>
  C: 8 -> 5, 6, 19, 29, 104, 105, 124, 127
  F: 8 -> 3, 4, 14, 17, 20, 22, 25, 125
  O: 8 -> 61, 65, 66, 69, 71, 75, 80, 82
  P: 7 -> 41, 51, 60, 79, 85, 94, 98
  TL: 5 -> 38, 39, 42, 50, 56
  TR: 5 -> 9

In [4]:
current_device = careful_pseudo_devices[0]

eeg_sample = train_dataset[0][0]
print(f"Original size: {eeg_sample.shape}")
transformed_sample = current_device.transform(eeg_sample)
print(f"Transformed sample shape (channel-last): {transformed_sample.shape}")

current_device_dataset = current_device.convert(train_dataset, drop_ref=True)
x_dev, *rest = current_device_dataset[0]
print(f"Shape after PseudoDeviceDataset conversion: {x_dev.shape}")

Original size: (129, 500)
Transformed sample shape (channel-last): (40, 500)
Shape after PseudoDeviceDataset conversion: torch.Size([40, 500])


In [ ]:
#data hyperparameters
patch_size = 20
patch_overlap = 5
window_sec = 5
mask_ratio = 0.7
mask_block = 4
train_batch_size = 128
valid_batch_size = 128

# model
encoder_depth = 3
emb_dim = 16

# training
lr = 1e-4
weight_decay = 1e-6

# logging
output_dir = "../../artefacts/models/embeddings"
experiment_name = "reconstuction_only_good_device_v2"


checkpoint_dir = os.path.join(output_dir, experiment_name)
os.makedirs(checkpoint_dir, exist_ok=True)

In [8]:
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader

from neurosned.foundation.reconstruction_dataset import ReconstructionPatchDataset


train_dev_datasets = [dev.convert(train_dataset, drop_ref=True) for dev in careful_pseudo_devices]
valid_dev_datasets = [dev.convert(valid_dataset, drop_ref=True) for dev in careful_pseudo_devices]

train_merged = ConcatDataset(train_dev_datasets)
valid_merged = ConcatDataset(valid_dev_datasets)

train_recon_merged = ReconstructionPatchDataset(
    base=train_merged,
    sfreq=100,
    window_sec=window_sec,
    patch_size=patch_size,
    patch_overlap=patch_overlap,
    pad_to_window=False,
    return_full=True,
)
valid_recon_merged = ReconstructionPatchDataset(
    base=valid_merged,
    sfreq=100,
    window_sec=window_sec,
    patch_size=patch_size,
    patch_overlap=patch_overlap,
    pad_to_window=False,
    return_full=True,
)

train_loader = DataLoader(train_recon_merged, batch_size=train_batch_size, shuffle=True, num_workers=4, collate_fn=train_recon_merged.collate_fn)
val_loader = DataLoader(valid_recon_merged, batch_size=valid_batch_size, shuffle=False, num_workers=4, collate_fn=valid_recon_merged.collate_fn)


In [14]:
import torch
from tqdm import tqdm

from neurosned.models.reconstruction.minimalistic import MinimalistReconstructor, make_block_mask
from neurosned.models.reconstruction.losses import reconstruction_loss


# 3) Model / optim
example_patches, _, _ = next(iter(train_loader))
n_channels = example_patches.shape[2]
patch_size = example_patches.shape[3]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MinimalistReconstructor(n_channels=n_channels, patch_size=patch_size, embed_dim=emb_dim, depth=encoder_depth).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

# 4) Training + validation loop (masked reconstruction)
import json

num_epochs = 5
metrics = {
    "epoch": [],
    "train_mse": [],
    "val_mse": [],
}
best_val_mse = float('inf')

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    n_batches = 0
    with tqdm(train_loader, desc=f"Epoch {epoch}", unit="batch") as tbar:
        for patches, starts, x_full in tbar:
            patches = patches.to(device)  # (B,P,C,T)
            B, P = patches.shape[:2]
            mask = make_block_mask(P, mask_ratio=mask_ratio, block_size=mask_block, device=patches.device)
            mask = mask.unsqueeze(0).expand(B, -1)  # (B,P)

            opt.zero_grad()
            out = model(patches, mask=mask)
            loss = reconstruction_loss(out["recon"], patches, mask=mask)
            loss.backward()
            opt.step()

            batch_loss = loss.item()
            running_loss += batch_loss
            n_batches += 1
            tbar.set_postfix(loss=f"{batch_loss:.4f}")

    train_mse = running_loss / max(1, n_batches)

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Val {epoch}", unit="batch") as vbar:
            for patches, starts, x_full in vbar:
                patches = patches.to(device)
                B, P = patches.shape[:2]
                mask = make_block_mask(P, mask_ratio=mask_ratio, block_size=mask_block, device=patches.device)
                mask = mask.unsqueeze(0).expand(B, -1)

                out = model(patches, mask=mask)
                loss = reconstruction_loss(out["recon"], patches, mask=mask)
                val_loss += loss.item()
                val_batches += 1
                vbar.set_postfix(val_loss=f"{loss.item():.4f}")
    val_mse = val_loss / max(1, val_batches)

    # Save metrics for this epoch
    metrics["epoch"].append(epoch)
    metrics["train_mse"].append(train_mse)
    metrics["val_mse"].append(val_mse)

    # Write metrics to file after each epoch
    metrics_path = os.path.join(checkpoint_dir, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    checkpoint_dict = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": opt.state_dict(),
            "val_mse": val_mse,
        }
    checkpoint_path = os.path.join(checkpoint_dir, "last.pt")
    torch.save(checkpoint_dict, checkpoint_path)
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        checkpoint_path = os.path.join(checkpoint_dir, "best.pt")
        torch.save(checkpoint_dict, checkpoint_path)
        print(f"Saved best checkpoint to {checkpoint_path} (val_mse={val_mse:.6f})")

    print(f"epoch {epoch}: train masked-MSE {train_mse:.6f} | val masked-MSE {val_mse:.6f}")


Val 1: 100%|██████████| 541/541 [00:13<00:00, 39.08batch/s, val_loss=0.2269]


Saved best checkpoint to ../../artefacts/models/embeddings/reconstuction_only_good_device_v2/best.pt (val_mse=0.276472)
epoch 1: train masked-MSE 28.225844 | val masked-MSE 0.276472


Val 2: 100%|██████████| 541/541 [00:13<00:00, 38.82batch/s, val_loss=0.0654]


Saved best checkpoint to ../../artefacts/models/embeddings/reconstuction_only_good_device_v2/best.pt (val_mse=0.089263)
epoch 2: train masked-MSE 1.023912 | val masked-MSE 0.089263


Val 3: 100%|██████████| 541/541 [00:13<00:00, 39.69batch/s, val_loss=0.0254]


Saved best checkpoint to ../../artefacts/models/embeddings/reconstuction_only_good_device_v2/best.pt (val_mse=0.049104)
epoch 3: train masked-MSE 0.270666 | val masked-MSE 0.049104


Val 4: 100%|██████████| 541/541 [00:13<00:00, 39.13batch/s, val_loss=0.0124]


Saved best checkpoint to ../../artefacts/models/embeddings/reconstuction_only_good_device_v2/best.pt (val_mse=0.033730)
epoch 4: train masked-MSE 0.094317 | val masked-MSE 0.033730


Val 5: 100%|██████████| 541/541 [00:13<00:00, 39.12batch/s, val_loss=0.0043]

Saved best checkpoint to ../../artefacts/models/embeddings/reconstuction_only_good_device_v2/best.pt (val_mse=0.026893)
epoch 5: train masked-MSE 0.049714 | val masked-MSE 0.026893


# Reload it back

In [11]:
import os, json, math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from neurosned.models.reconstruction.minimalistic import MinimalistReconstructor, make_block_mask

example_patches, _, _ = train_recon_merged[0]  # уже patchified ранее
n_channels, patch_size_used = example_patches.shape[1], example_patches.shape[2]

enc = MinimalistReconstructor(n_channels=n_channels, patch_size=patch_size, embed_dim=emb_dim, depth=encoder_depth)
state =  torch.load(os.path.join(checkpoint_dir, "best.pt"))['model_state']
enc.load_state_dict(state)
enc.eval()
for p in enc.parameters():
    p.requires_grad_(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
enc.to(device)

MinimalistReconstructor(
  (patch_proj): Linear(in_features=800, out_features=16, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
        )
        (linear1): Linear(in_features=16, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=32, out_features=16, bias=True)
        (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (recon_proj): Linear(in_features=16, out_features=800, bias=True)
)

In [27]:
class ReconWithLabel(Dataset):
    def __init__(self, base, device_id, **patch_kwargs):
        self.base = base
        self.device_id = device_id
        self.patch_ds = ReconstructionPatchDataset(base, return_full=False, **patch_kwargs)

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        patches, starts = self.patch_ds[idx]           # (P,C,T), (P,)
        sample = self.base[idx]
        y = sample[1] if isinstance(sample, (tuple, list)) else None
        return patches, torch.as_tensor(y, dtype=torch.float32), self.device_id

    @staticmethod
    def collate_fn(batch):
        patches, y, dev = zip(*batch)
        return (
            torch.stack(patches),          # (B,P,C,T)
            torch.stack(y),                # (B,)
            dev,                           # list of device ids
        )

# --- 3) датасеты по девайсам ---
def make_ds(base_ds, dev_obj, dev_name):
    wrapped = dev_obj.convert(base_ds, drop_ref=True)
    return ReconWithLabel(
        wrapped,
        device_id=dev_name,
        sfreq=100,
        window_sec=window_sec,
        patch_size=patch_size,
        patch_overlap=patch_overlap,
        pad_to_window=False,
    )

train_ds_S1 = make_ds(train_dataset, careful_pseudo_devices[0], "S1")
val_ds_S1   = make_ds(valid_dataset, careful_pseudo_devices[0], "S1")  # ID
val_ds_S2   = make_ds(valid_dataset, careful_pseudo_devices[1], "S2")  # OOD
val_ds_S3   = make_ds(valid_dataset, careful_pseudo_devices[2], "S3")  # OOD

def make_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=4, collate_fn=ds.collate_fn)

batch_train, batch_eval = 64, 128
train_loader = make_loader(train_ds_S1, batch_train, True)
val_loaders = {
    "S1_ID": make_loader(val_ds_S1, batch_eval, False),
    "S2_OOD": make_loader(val_ds_S2, batch_eval, False),
    "S3_OOD": make_loader(val_ds_S3, batch_eval, False),
}

# Extract embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(loader):
    embs, ys, devs = [], [], []
    for patches, y, dev in tqdm(loader, leave=False):
        patches = patches.to(device)
        z = enc(patches, mask=None)["z_global"]  # (B,D)
        embs.append(z.cpu())
        ys.append(y)
        devs.extend(dev)
    return torch.cat(embs), torch.cat(ys), devs

train_emb, train_y, _ = extract_embeddings(train_loader)
val_data = {name: extract_embeddings(l) for name, l in val_loaders.items()}

# --- 5) маленькая регрессия ---
class Head(nn.Module):
    def __init__(self, dim, hidden=128, p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(p),
            nn.Linear(hidden, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

head = Head(enc.embed_dim).to(device)
opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

def rmse(pred, target):
    return math.sqrt(loss_fn(pred, target).item())


In [31]:
train_emb.shape

torch.Size([92845, 16])

In [30]:
train_emb.shape

torch.Size([92845, 16])

In [ ]:
# --- 6) обучение головы ---
num_epochs = 20
best = None
for epoch in range(1, num_epochs + 1):
    head.train()
    opt.zero_grad()
    pred = head(train_emb.to(device))
    loss = loss_fn(pred, train_y.to(device))
    loss.backward()
    opt.step()

    head.eval()
    with torch.no_grad():
        metrics = {}
        pred_id = head(val_data["S1_ID"][0].to(device))
        metrics["rmse_id"] = rmse(pred_id, val_data["S1_ID"][1].to(device))
        for k in ["S2_OOD", "S3_OOD"]:
            p = head(val_data[k][0].to(device))
            metrics[f"rmse_{k.lower()}"] = rmse(p, val_data[k][1].to(device))
        metrics["gap_rmse"] = ((metrics["rmse_s2_ood"] + metrics["rmse_s3_ood"]) / 2) - metrics["rmse_id"]

    print(f"epoch {epoch}: train_loss={loss.item():.4f} | "
          f"RMSE_ID={metrics['rmse_id']:.4f} | "
          f"S2={metrics['rmse_s2_ood']:.4f} | S3={metrics['rmse_s3_ood']:.4f} | "
          f"gap={metrics['gap_rmse']:.4f}")

    if best is None or metrics["gap_rmse"] < best["gap_rmse"]:
        best = {"epoch": epoch, "train_loss": loss.item(), **metrics}

print("Best:", best)